In [1]:
USE_KAGGLE_SANDBOX = True
KAGGLE_SANDBOX_KWARGS = {
    'dataset_data_sources': ['greynearing/camels-us-531-era'],
    'enable_gpu': True,
    'enable_internet': False,
    'machine_shape': 'NvidiaTeslaT4Highmem',
}
C_PUCT = 2.5

# CAMELS-US ERA — PUB (12-Fold Random) Minimal Guidance

## Objective
Maximize the **out-of-sample Median Nash-Sutcliffe Efficiency (NSE)** across all **531 CAMELS-US basins** under the **12-Fold Randomized Prediction in Ungauged Basins (PUB)** cross-validation protocol on the evaluation period (`1999-10-01` to `2003-09-30`, 4 water years, `1,461` days).

## Data & Split Protocol
- **Spatial Regime (12-Fold PUB Cross-Validation)**:
  - `CV_FOLDS` (loaded from `pub_12fold_random_splits.json`) partitions the 531 basins into 12 mutually exclusive test folds (`~44-45` test basins and `~486-487` training basins per fold).
  - In each fold `k`, you **must** train exclusively on `fold['train_basin_mask']` using `fold['qobs_fold_train']` (`2003-10-01` to `2008-09-30`), where all test basins in `fold['test_basin_mask']` are masked to `NaN`.
  - Predict out-of-sample discharge for `fold['test_idx']` and assemble the full `Q_PRED` array of shape `(531, 3653)` across all 12 folds.
- **Strict Newman Temporal Containment (`1998-10-01` to `2008-09-30`, `3,653` days)**:
  - `1998-10-01` to `1999-09-30` (`365` days): 1-year meteorological spin-up prior to evaluation (`QOBS_TRAIN = NaN`).
  - `1999-10-01` to `2003-09-30` (`1,461` days, `ERA_EVAL_MASK`): Out-of-sample ERA hill-climb evaluation period (`QOBS_TRAIN = NaN`).
  - `2003-10-01` to `2008-09-30` (`1,827` days, `ERA_TRAIN_MASK`): ERA model weights training period on each fold's training basins.
- **Inputs Available**:
  - `DYNAMIC_INPUTS`: shape `(531, 3653, 15)` (`float32`) — 5 daily meteorological variables (`prcp`, `srad`, `tmax`, `tmin`, `vp`) across 3 forcing products (`Daymet`, `Maurer`, `NLDAS`).
  - `AUX_FORCINGS`: shape `(531, 3653, 6)` (`float32`) — daily `dayl` and `swe` across `Daymet`, `Maurer`, `NLDAS`.
  - `STATIC_ATTRIBUTES`: shape `(531, 27)` (`float32`) — 27 catchment climate, topography, soil, geology, and vegetation attributes (plus `BASIN_AREAS`, `LAT`, `LON`, `HUC_02`).
- **Required Output**:
  - The mutable cell must produce `Q_PRED` as a `float32` NumPy array of shape `(531, 3653)` containing daily streamflow predictions (`mm/day`).

## Modeling & Exploration Scope
You are free to explore any machine learning architecture, sequence length, feature representation, normalization strategy, loss function, regularization, optimizer schedule, or multi-model ensembling strategy, provided all predictions strictly respect the causal temporal masks and spatial fold boundaries.

In [2]:
import os
import time
import math
import json
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Ensure reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

_SEARCH_DIRS = [
    Path("/kaggle/input/camels-us-531-era"),
    Path("/kaggle/input/datasets/greynearing/camels-us-531-era"),
    Path(__file__).resolve().parent.parent / "splits" if "__file__" in globals() else Path("../splits"),
    Path("../splits"),
    Path("./splits"),
]

def _find_file(filename: str) -> Path:
    for d in _SEARCH_DIRS:
        candidate = d / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {filename} in any of {_SEARCH_DIRS}")

_train_npz = np.load(_find_file("camels_era_train.npz"), allow_pickle=True)
_basins_json = json.loads(_find_file("basins_531.json").read_text(encoding="utf-8"))

BASINS = _train_npz["basins"].astype(str)
_canonical_basins = [str(b) for b in _basins_json]
assert list(BASINS) == _canonical_basins, "BASINS order mismatch against canonical basins_531.json"

# Strictly enforce containment to the Newman training window (1998-10-01 to 2008-09-30, 3653 days)
_raw_dates = np.array([str(d) for d in _train_npz["dates"]])
_newman_window = (_raw_dates >= "1998-10-01") & (_raw_dates <= "2008-09-30")
DATES = _raw_dates[_newman_window]
assert len(DATES) == 3653 and DATES[0] == "1998-10-01" and DATES[-1] == "2008-09-30"

DYNAMIC_INPUTS = _train_npz["dynamic_inputs"][:, _newman_window, :].astype(np.float32)
DYNAMIC_FEATURE_NAMES = [str(x) for x in _train_npz["dynamic_feature_names"]]
AUX_FORCINGS = _train_npz["aux_forcings"][:, _newman_window, :].astype(np.float32)
AUX_FEATURE_NAMES = [str(x) for x in _train_npz["aux_feature_names"]]
STATIC_ATTRIBUTES = _train_npz["static_attributes"].astype(np.float32)
STATIC_ATTRIBUTE_NAMES = [str(x) for x in _train_npz["static_attribute_names"]]

ERA_TRAIN_MASK = (DATES >= "2003-10-01") & (DATES <= "2008-09-30")
ERA_EVAL_MASK = (DATES >= "1999-10-01") & (DATES <= "2003-09-30")
assert int(ERA_TRAIN_MASK.sum()) == 1827, f"Expected 1827 training days, got {int(ERA_TRAIN_MASK.sum())}"
assert int(ERA_EVAL_MASK.sum()) == 1461, f"Expected 1461 eval days, got {int(ERA_EVAL_MASK.sum())}"

QOBS_TRAIN = np.where(
    ERA_TRAIN_MASK[None, :],
    _train_npz["qobs_train"][:, _newman_window].astype(np.float32),
    np.nan,
).astype(np.float32)

BASIN_AREAS = _train_npz["basin_areas"].astype(np.float32)
LAT = _train_npz["lat"].astype(np.float32)
LON = _train_npz["lon"].astype(np.float32)

SPATIAL_REGIME = "pub_12fold"
_splits_json = json.loads(_find_file("pub_12fold_random_splits.json").read_text(encoding="utf-8"))
CV_SPLITS = [v for _, v in sorted(_splits_json["folds"].items(), key=lambda kv: int(kv[1]["fold_index"]))]
assert len(CV_SPLITS) == 12, f"Expected 12 PUB folds, got {len(CV_SPLITS)}"

_basin_to_idx = {str(b): i for i, b in enumerate(BASINS)}
CV_FOLDS = []
_covered_test = np.zeros(len(BASINS), dtype=bool)
for _fold in CV_SPLITS:
    _fid = int(_fold["fold_index"])
    _test_idx = np.array([_basin_to_idx[str(b)] for b in _fold["test_basins"]], dtype=np.int64)
    _train_idx = np.array([_basin_to_idx[str(b)] for b in _fold["train_basins"]], dtype=np.int64)
    _test_mask = np.zeros(len(BASINS), dtype=bool)
    _test_mask[_test_idx] = True
    _train_mask = np.zeros(len(BASINS), dtype=bool)
    _train_mask[_train_idx] = True
    assert not np.any(_test_mask & _train_mask), f"Train/test overlap in PUB fold {_fid}"
    assert not np.any(_covered_test & _test_mask), f"Duplicate test basins in PUB fold {_fid}"
    _covered_test |= _test_mask
    CV_FOLDS.append({
        "fold_id": _fid,
        "train_basins": [str(b) for b in _fold["train_basins"]],
        "test_basins": [str(b) for b in _fold["test_basins"]],
        "train_idx": _train_idx,
        "test_idx": _test_idx,
        "train_basin_mask": _train_mask,
        "test_basin_mask": _test_mask,
        "qobs_fold_train": np.where(
            _train_mask[:, None] & ERA_TRAIN_MASK[None, :], QOBS_TRAIN, np.nan
        ).astype(np.float32),
    })
assert np.all(_covered_test), "12 PUB folds do not cover all 531 basins!"

print(f"Loaded CAMELS-US 531 dataset ({SPATIAL_REGIME}): {len(BASINS)} basins, {len(DATES)} days ({DATES[0]} to {DATES[-1]}), {len(CV_FOLDS)} folds.")
print(f"ERA_TRAIN_MASK (2003-10-01..2008-09-30): {int(ERA_TRAIN_MASK.sum())} days | ERA_EVAL_MASK (1999-10-01..2003-09-30): {int(ERA_EVAL_MASK.sum())} days")
print(f"Dynamic inputs: {DYNAMIC_INPUTS.shape}, Aux: {AUX_FORCINGS.shape}, Static: {STATIC_ATTRIBUTES.shape}")
print(f"Available GPUs: {NUM_GPUS} (Primary device: {DEVICE})")

Loaded CAMELS-US 531 dataset (pub_12fold): 531 basins, 3653 days (1998-10-01 to 2008-09-30), 12 folds.
ERA_TRAIN_MASK (2003-10-01..2008-09-30): 1827 days | ERA_EVAL_MASK (1999-10-01..2003-09-30): 1461 days
Dynamic inputs: (531, 3653, 15), Aux: (531, 3653, 6), Static: (531, 27)
Available GPUs: 2 (Primary device: cuda)


In [3]:
# Begin Stage

In [4]:
INSTRUCTION = [
    "To maximize out-of-sample generalization to completely ungauged basins across all 12 folds, implement static-attribute dropout / Gaussian noise regularization on the 27 static catchment attributes during training so the network learns smooth, interpolatable representations rather than memorizing individual training basin IDs.",
    "Design a Hypernetwork or Feature-wise Linear Modulation (FiLM) conditioning module where a regularized MLP maps static catchment attributes (with dropout and L2 weight decay) to scale-and-shift modulations of the sequence model layers, improving zero-shot transfer to held-out test basins.",
    "Engineer domain-generalizable dimensionless and multi-scale dynamic features (rolling cumulative precipitation/radiation ratios, exponentially weighted antecedent input indices over 3, 7, 15, 30, 60, 120, 365 days, and inter-product forcing spread across Daymet/Maurer/NLDAS) that invariant-scale across basins.",
    "Explore a multi-architecture ensemble per fold (e.g., combining a FiLM-modulated LSTM, a gated GRU with residual temporal convolutions, or a multi-scale sequence network) using mixed-precision GPU training and parallel multi-GPU fold execution (`torch.cuda.device_count()`) to train all 12 folds efficiently.",
    "Improve target and loss scaling for ungauged transfer: since test basins are unseen during training, explore training with a composite loss combining basin-normalized NSE (using training-basin stds), log-transformed MSE (`log(Q + eps)`), and relative volume bias penalties so predictions remain well-calibrated across both arid and humid ungauged catchments.",
    "Use Stochastic Weight Averaging (SWA) or Exponential Moving Average (EMA) of model weights during the final training epochs of each fold, combined with cosine annealing learning rate schedule and input feature noise, to flatten the loss landscape and boost out-of-sample basin generalization.",
    "Explore low-rank bottleneck embeddings for static catchment attributes (compressing the 27 noisy static features into a compact, L2-normalized catchment representation before concatenating or modulating dynamic inputs) to prevent overfitting to collinear static descriptors.",
    "Synthesize the strongest ungauged generalization mechanisms\u2014regularized static bottleneck + FiLM modulation, multi-window temporal context features, composite NSE + log-MSE loss, EMA weight averaging, and multi-seed/multi-head ensembling across all 12 folds\u2014into a fast, cohesive 12-fold pipeline."
]
VALUE_REGEX = r'(?m)^Candidate metric to be maximized: (.*)$'
OBJECTIVE = 'maximize'
WORST_VALUE = -2e308

In [5]:
# [rewrite_cell]
# ==============================================================================
# Enhanced EA-LSTM for Randomized PUB (12-Fold)
# ==============================================================================

import copy

# Hyperparameters
SEQ_LEN = 365
HIDDEN_SIZE = 256
STAT_EMB_SIZE = 128
DROPOUT = 0.3
BATCH_SIZE = 256
LR = 2.5e-3
WEIGHT_DECAY = 1e-2
EPOCHS_PER_FOLD = 45
SAMPLES_PER_BASIN_PER_EPOCH = 64
EMA_ALPHA = 0.999
INPUT_NOISE_STD = 0.005

# 1. Feature Engineering (Pre-calculated once)
t_range = DYNAMIC_INPUTS[:, :, 6:9] - DYNAMIC_INPUTS[:, :, 9:12]
d_dates = pd.to_datetime(DATES)
doy = d_dates.dayofyear.values
doy_sin = np.tile(np.sin(2 * np.pi * doy / 365.25).astype(np.float32)[None, :, None], (len(BASINS), 1, 1))
doy_cos = np.tile(np.cos(2 * np.pi * doy / 365.25).astype(np.float32)[None, :, None], (len(BASINS), 1, 1))

X_dyn_raw = np.concatenate([DYNAMIC_INPUTS, AUX_FORCINGS, t_range, doy_sin, doy_cos], axis=-1)
log_area = np.log1p(BASIN_AREAS[:, None])
X_stat_raw = np.concatenate([STATIC_ATTRIBUTES, LAT[:, None], LON[:, None], log_area], axis=-1)

class EALSTM(nn.Module):
    def __init__(self, dyn_dim, stat_dim, hidden_size, stat_emb_size, dropout):
        super().__init__()
        self.stat_embed = nn.Sequential(
            nn.Linear(stat_dim, stat_emb_size),
            nn.LayerNorm(stat_emb_size),
            nn.SiLU(),
            nn.Linear(stat_emb_size, stat_emb_size),
            nn.LayerNorm(stat_emb_size),
            nn.SiLU(),
            nn.Dropout(dropout)
        )
        self.lstm = nn.LSTM(dyn_dim + stat_emb_size, hidden_size, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )
        
    def forward(self, x_dyn, x_stat, return_seq=False):
        s_emb = self.stat_embed(x_stat) # (B, E)
        s_rep = s_emb.unsqueeze(1).expand(-1, x_dyn.size(1), -1) # (B, L, E)
        x_combined = torch.cat([x_dyn, s_rep], dim=-1)
        
        # Flatten parameters for CuDNN efficiency
        self.lstm.flatten_parameters()
        out, _ = self.lstm(x_combined)
        
        if return_seq:
            return self.head(out).squeeze(-1)
        return self.head(out[:, -1, :]).squeeze(-1)

def hybrid_nse_loss(y_hat, y_true, b_std):
    """Combines basin-scaled Huber loss and log-MSE for PUB robustness."""
    # Scale error by basin standard deviation (mimics NSE optimization)
    scaled_err = (y_hat - y_true) / (b_std + 0.1)
    # Huber loss is more robust to outliers in poor-performing basins
    huber = F.huber_loss(scaled_err, torch.zeros_like(scaled_err), delta=1.0)
    # Log-domain loss captures low-flow performance
    log_mse = torch.mean(torch.square(torch.log1p(torch.relu(y_hat)) - torch.log1p(torch.relu(y_true))))
    return huber + 0.1 * log_mse

def train_and_predict_fold(fold, device_id):
    dev = torch.device(f"cuda:{device_id}")
    f_id = fold["fold_id"]
    train_mask = fold["train_basin_mask"]
    test_idx = fold["test_idx"]
    qobs = fold["qobs_fold_train"]

    # Norm Stats (PUB: Train basins only)
    d_mean = X_dyn_raw[train_mask][:, ERA_TRAIN_MASK].mean(axis=(0, 1))
    d_std = X_dyn_raw[train_mask][:, ERA_TRAIN_MASK].std(axis=(0, 1)) + 1e-6
    X_d_norm = ((X_dyn_raw - d_mean) / d_std).astype(np.float32)

    s_train = X_stat_raw[train_mask]
    s_mean, s_std = np.nanmean(s_train, axis=0), np.nanstd(s_train, axis=0) + 1e-6
    X_s_proc = np.where(np.isnan(X_stat_raw), s_mean, X_stat_raw)
    X_s_norm = ((X_s_proc - s_mean) / s_std).astype(np.float32)

    y_train_vals = qobs[train_mask][:, ERA_TRAIN_MASK]
    y_m, y_s = np.nanmean(y_train_vals), np.nanstd(y_train_vals) + 1e-6
    Y_norm = ((qobs - y_m) / y_s).astype(np.float32)

    b_stds = np.ones(len(BASINS), dtype=np.float32)
    valid_t_map = {}
    for b in fold["train_idx"]:
        q_b = qobs[b, ERA_TRAIN_MASK]
        q_b_valid = q_b[~np.isnan(q_b)]
        if len(q_b_valid) > 90:
            b_stds[b] = max(float(np.std(q_b_valid)), 0.02) / y_s
            t_idx = np.where(ERA_TRAIN_MASK & ~np.isnan(qobs[b]))[0]
            t_idx = t_idx[t_idx >= SEQ_LEN - 1]
            if len(t_idx) > 0: valid_t_map[b] = t_idx

    Xd_g, Xs_g, Y_g, Bstd_g = [torch.from_numpy(a).to(dev) for a in [X_d_norm, X_s_norm, Y_norm, b_stds]]
    
    model = EALSTM(X_d_norm.shape[-1], X_s_norm.shape[-1], HIDDEN_SIZE, STAT_EMB_SIZE, DROPOUT).to(dev)
    ema_model = copy.deepcopy(model).eval()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    
    # Calculate total steps for OneCycleLR
    num_samples = len(valid_t_map) * SAMPLES_PER_BASIN_PER_EPOCH
    steps_per_epoch = math.ceil(num_samples / BATCH_SIZE)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, steps_per_epoch=steps_per_epoch, epochs=EPOCHS_PER_FOLD)
    
    rng = np.random.RandomState(SEED + f_id)
    offsets = torch.arange(-SEQ_LEN + 1, 1, device=dev).unsqueeze(0)

    for epoch in range(EPOCHS_PER_FOLD):
        model.train()
        pb, pt = [], []
        for b, t_arr in valid_t_map.items():
            idx = rng.choice(t_arr, size=SAMPLES_PER_BASIN_PER_EPOCH, replace=True)
            pb.append(np.full(len(idx), b, dtype=np.int64)); pt.append(idx)
        
        pb, pt = np.concatenate(pb), np.concatenate(pt)
        indices = rng.permutation(len(pb))
        
        for i in range(0, len(pb), BATCH_SIZE):
            batch_idx = indices[i:i+BATCH_SIZE]
            ib, it = torch.from_numpy(pb[batch_idx]).to(dev), torch.from_numpy(pt[batch_idx]).to(dev)
            
            x_d_batch = Xd_g[ib.unsqueeze(1), it.unsqueeze(1) + offsets]
            if INPUT_NOISE_STD > 0:
                x_d_batch = x_d_batch + torch.randn_like(x_d_batch) * INPUT_NOISE_STD
            
            opt.zero_grad()
            pred = model(x_d_batch, Xs_g[ib])
            loss = hybrid_nse_loss(pred, Y_g[ib, it], Bstd_g[ib])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            
            with torch.no_grad():
                for p, ep in zip(model.parameters(), ema_model.parameters()):
                    ep.data.mul_(EMA_ALPHA).add_(p.data, alpha=1-EMA_ALPHA)

    with torch.no_grad():
        ema_model.eval()
        t_gpu = torch.from_numpy(test_idx).to(dev)
        p_norm = ema_model(Xd_g[t_gpu], Xs_g[t_gpu], return_seq=True)
        p_mm = torch.clamp(p_norm * y_s + y_m, min=0.0).cpu().numpy()
        
    return test_idx, p_mm

Q_PRED = np.zeros((len(BASINS), len(DATES)), dtype=np.float32)
n_workers = max(1, NUM_GPUS)
with ThreadPoolExecutor(max_workers=n_workers) as ex:
    futures = [ex.submit(train_and_predict_fold, f, i % n_workers) for i, f in enumerate(CV_FOLDS)]
    for fut in futures:
        idx, pred = fut.result()
        Q_PRED[idx] = pred

In [6]:
# ==============================================================================
# Immutable ERA Hill-Climb Evaluation (Strictly 1999-10-01 to 2003-09-30)
# ==============================================================================
_eval_npz = np.load(_find_file("era_eval_targets.npz"), allow_pickle=True)
_eval_raw_dates = np.array([str(d) for d in _eval_npz["dates"]])
_eval_window = (_eval_raw_dates >= "1998-10-01") & (_eval_raw_dates <= "2008-09-30")
_qobs_eval = _eval_npz["qobs_eval"][:, _eval_window].astype(np.float32)

assert "Q_PRED" in globals(), "Mutable cell must define Q_PRED!"
Q_PRED = np.asarray(Q_PRED, dtype=np.float32)
assert Q_PRED.shape == (len(BASINS), len(DATES)), (
    f"Expected Q_PRED shape {(len(BASINS), len(DATES))}, got {Q_PRED.shape}"
)
assert not np.any(np.isnan(Q_PRED[:, ERA_EVAL_MASK])) and not np.any(np.isinf(Q_PRED[:, ERA_EVAL_MASK])), (
    "Q_PRED contains NaN or Inf on ERA_EVAL_MASK (1999-10-01 to 2003-09-30)!"
)

def _compute_basin_nses(q_pred: np.ndarray, q_true: np.ndarray, mask: np.ndarray) -> np.ndarray:
    nses = []
    for i in range(len(BASINS)):
        yt = q_true[i, mask]
        yp = q_pred[i, mask]
        valid = ~np.isnan(yt)
        if np.sum(valid) < 30:
            continue
        yt_v = yt[valid]
        yp_v = yp[valid]
        denom = np.sum((yt_v - np.mean(yt_v)) ** 2)
        if denom > 1e-12:
            nses.append(1.0 - float(np.sum((yp_v - yt_v) ** 2) / denom))
    return np.asarray(nses, dtype=np.float64)

_nses_eval = _compute_basin_nses(Q_PRED, _qobs_eval, ERA_EVAL_MASK)
_median_nse = float(np.median(_nses_eval))
_mean_nse = float(np.mean(_nses_eval))
_q25, _q75 = float(np.percentile(_nses_eval, 25)), float(np.percentile(_nses_eval, 75))
_neg_count = int(np.sum(_nses_eval < 0.0))

print(f"=== ERA Evaluation Summary ({SPATIAL_REGIME} | {len(_nses_eval)} Basins | 1999-10-01 to 2003-09-30) ===")
print(f"Median NSE (1999-2003): {_median_nse:.5f} | Mean NSE: {_mean_nse:.5f} | IQR: [{_q25:.5f}, {_q75:.5f}] | Basins < 0: {_neg_count}")
print(f"Candidate metric to be maximized: {_median_nse:.6f}")

=== ERA Evaluation Summary (pub_12fold | 531 Basins | 1999-10-01 to 2003-09-30) ===
Median NSE (1999-2003): 0.73035 | Mean NSE: 0.47607 | IQR: [0.54878, 0.80718] | Basins < 0: 23
Candidate metric to be maximized: 0.730350


In [7]:
# End Stage